# Med-DDPM v2 — Conditional Diffusion for Brain MRI Synthesis
Based on: Dorjsembe et al. 2024 — Conditional Diffusion Models for Semantic 3D Brain MRI Synthesis

Adapted from 3D → 2D for LGG MRI slices (256×256×3). Mask conditioning via channel-wise concatenation at every denoising step.

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone or update the repository
import os

REPO_URL = "https://github.com/AmineAitLaamim/Mask-to-MRI.git"
PROJECT_DIR = "/content/Mask-to-MRI"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}
    print("✅ Repository cloned.")
else:
    %cd $PROJECT_DIR
    !git pull
    print("✅ Repository updated.")

In [ ]:
# Install dependencies
%cd $PROJECT_DIR
!pip install -q torch torchvision albumentations tifffile scikit-image pyyaml tqdm matplotlib

In [ ]:
# ── Mount Drive (persists checkpoints across disconnects) ──
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Create local output directories (no symlinks — train.py handles Drive sync)
for d in ["checkpoints", "samples", "metrics", "synthetic"]:
    os.makedirs(os.path.join(PROJECT_DIR, "outputs_v2", d), exist_ok=True)

print("✅ Drive mounted. Output directories created.")

## 2. Upload Dataset

In [ ]:
# ── Optimized Dataset Setup ──
import os, shutil

os.makedirs(os.path.join(PROJECT_DIR, "dataset"), exist_ok=True)

local_folder = os.path.join(PROJECT_DIR, "dataset", "lgg-mri-segmentation")
zip_local = '/content/lgg-mri-segmentation.zip'

# Clean up old link/folder
if os.path.islink(local_folder):
    os.remove(local_folder)
elif os.path.isdir(local_folder):
    shutil.rmtree(local_folder)

# Extract only if local folder doesn't exist
if not os.path.exists(local_folder):
    # Find zip in Drive
    zip_drive = None
    for candidate in [
        '/content/drive/MyDrive/mask-to-mri/dataset/lgg-mri-segmentation.zip',
        '/content/drive/My Drive/mask-to-mri/dataset/lgg-mri-segmentation.zip',
        os.path.join(PROJECT_DIR, 'dataset', 'lgg-mri-segmentation.zip'),
    ]:
        if os.path.exists(candidate):
            zip_drive = candidate
            break

    if zip_drive:
        print(f"Found zip: {zip_drive}")
        if not os.path.exists(zip_local):
            print("Copying zip to local disk...")
            shutil.copy2(zip_drive, zip_local)
            print("Copy complete.")
        print("Unzipping to local disk...")
        !unzip -q -o "$zip_local" -d /content/
        print("Unzip complete.")
        if os.path.exists('/content/lgg-mri-segmentation'):
            shutil.move('/content/lgg-mri-segmentation', local_folder)
            print("✅ Dataset extracted and moved.")
    else:
        print("⚠️  Zip not found. Upload to Drive at MyDrive/mask-to-mri/dataset/")
elif os.path.exists(local_folder):
    print("✅ Dataset already extracted.")

print(f"   Path: {PROJECT_DIR}/dataset/lgg-mri-segmentation/")

## 3. Import Modules & Load Config

In [ ]:
# ── Force clear any cached src modules ──
import sys
for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, PROJECT_DIR)

from src.med_ddpm_v2 import ConditionalDDPM, CONFIG
from src.med_ddpm_v2.train import train
from src.dataset import build_dataloaders

# Override timesteps to full noise schedule
CONFIG["timesteps"] = 1000

print("✅ Config loaded:")
print(f"   Timesteps:      {CONFIG['timesteps']}")
print(f"   Epochs:         {CONFIG['epochs']}")
print(f"   Batch size:     {CONFIG['batch_size']}")
print(f"   LR:             {CONFIG['lr']}")
print(f"   EMA decay:      {CONFIG['ema_decay']}")
print(f"   DDIM steps:     {CONFIG['ddim_steps']}")

## 4. Initialize Model & Data Loaders

In [ ]:
import torch
import random
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# Fix seeds
seed = CONFIG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
print(f"✅ Random seed fixed: {seed}")

In [ ]:
# Build DataLoaders — num_workers=0 on Colab (T4 has only 2 vCPUs)
loaders = build_dataloaders(
    CONFIG["raw_dir"],
    batch_size=CONFIG["batch_size"],
    num_workers=0,
    tumor_ratio=CONFIG["tumor_ratio"],
    seed=CONFIG["seed"],
)

print(f"✅ DataLoaders created:")
print(f"   Train: {len(loaders['train'].dataset)} samples")
print(f"   Val:   {len(loaders['val'].dataset)} samples")
print(f"   Batches/epoch (train): {len(loaders['train'])}")

In [ ]:
# Verify a batch
mask_batch, mri_batch = next(iter(loaders["train"]))
print(f"✅ Batch shape: mask={mask_batch.shape}, mri={mri_batch.shape}")
print(f"Channel 0 (R/T1)    mean: {mri_batch[:,0].mean():.3f}, std: {mri_batch[:,0].std():.3f}")
print(f"Channel 1 (G/FLAIR) mean: {mri_batch[:,1].mean():.3f}, std: {mri_batch[:,1].std():.3f}")
print(f"Channel 2 (B/T2)    mean: {mri_batch[:,2].mean():.3f}, std: {mri_batch[:,2].std():.3f}")

In [ ]:
# Create model
model = ConditionalDDPM(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model created: {n_params:,} parameters ({n_params/1e6:.2f}M)")

## 5. Pre-Training Sanity Check

In [ ]:
print("=" * 60)
print("PRE-TRAINING SANITY CHECK")
print("=" * 60)

# 1. DataLoader batch (normalized)
print("\n📦 DATALOADER BATCH (normalized):")
mask_batch, mri_batch = next(iter(loaders["train"]))
print(f"  Batch shape: mask={mask_batch.shape}, mri={mri_batch.shape}")
print(f"  mri min={mri_batch.min():.3f}, max={mri_batch.max():.3f}, std={mri_batch.std():.3f}")
print(f"  Ch0 (R/T1)    mean: {mri_batch[:,0].mean():+.3f}, std: {mri_batch[:,0].std():.3f}")
print(f"  Ch1 (G/FLAIR) mean: {mri_batch[:,1].mean():+.3f}, std: {mri_batch[:,1].std():.3f}")
print(f"  Ch2 (B/T2)    mean: {mri_batch[:,2].mean():+.3f}, std: {mri_batch[:,2].std():.3f}")

# 2. Check mask values
print("\n🎯 MASK VALUES:")
import numpy as np
unique_vals = np.unique(mask_batch.numpy())
print(f"  min={mask_batch.min():.3f}, max={mask_batch.max():.3f}, mean={mask_batch.mean():.3f}")
print(f"  Unique values: {unique_vals[:10]}..." if len(unique_vals) > 10 else f"  Unique values: {unique_vals}")

# 3. Quick forward pass
print("\n🔬 FORWARD PASS (1 batch):")
mask_d = mask_batch[:2].to(device)
mri_d = mri_batch[:2].to(device)
loss = model(mri_d, mask_d)
print(f"  Loss: {loss.item():.4f}")

# 4. Quick sample (10 DDIM steps)
print("\n🎨 SAMPLE (10 DDIM steps, 1 image):")
import time
start = time.time()
with torch.no_grad():
    fake = model.sample(mask_d[:1], ddim_steps=10)
elapsed = time.time() - start
print(f"  10 steps took: {elapsed:.1f}s")
print(f"  Output: min={fake.min():.3f}, max={fake.max():.3f}, std={fake.std():.3f}")
print(f"  Full 250 steps estimated: {elapsed * 25:.0f}s")

# 5. Save debug images
print("\n💾 SAVING DEBUG IMAGES:")
from PIL import Image
mask_np = ((mask_d[0, 0].cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
mri_np = ((mri_d[0].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
fake_np = ((fake[0].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

Image.fromarray(mask_np).save("debug_mask.png")
Image.fromarray(mri_np).save("debug_real_mri.png")
Image.fromarray(fake_np).save("debug_partial_sample.png")
print("  Saved: debug_mask.png, debug_real_mri.png, debug_partial_sample.png")

print("\n" + "=" * 60)
print("✅ SANITY CHECK COMPLETE — review values above before training")
print("=" * 60)

## 6. Restore Checkpoints from Drive

In [ ]:
# Restore checkpoints from Drive (survives Colab disconnects)
import shutil, os, glob

local_ckpt = "/content/Mask-to-MRI/outputs_v2/checkpoints"
drive_ckpt = "/content/drive/MyDrive/mask-to-mri/outputs_v2/checkpoints"

if os.path.exists(drive_ckpt):
    shutil.copytree(drive_ckpt, local_ckpt, dirs_exist_ok=True)
    found = glob.glob(f"{local_ckpt}/checkpoint_v2_epoch_*.pt")
    print(f"✅ Restored {len(found)} checkpoints from Drive")
else:
    print("No Drive checkpoints found")

## 7. Check for Resumable Checkpoint

In [ ]:
import glob
from pathlib import Path

checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint_v2_epoch_*.pt")

# Set START_FRESH = True to restart from epoch 0
START_FRESH = False

# To resume from a specific checkpoint:
# SPECIFIC_RESUME = f"{CONFIG['checkpoint_dir']}/checkpoint_v2_epoch_40.pt"
SPECIFIC_RESUME = None

if SPECIFIC_RESUME:
    print(f"Resuming from specific checkpoint: {SPECIFIC_RESUME}")
    resume_from = SPECIFIC_RESUME
elif checkpoints and not START_FRESH:
    latest = max(checkpoints, key=lambda p: int(Path(p).stem.split('_')[-1]))
    print(f"Resuming from checkpoint: {latest}")
    resume_from = latest
else:
    if START_FRESH and checkpoints:
        for ckpt in checkpoints:
            os.remove(ckpt)
        print("Fresh start — old checkpoints deleted.")
    print("No checkpoint found. Starting from scratch.")
    resume_from = None

## 8. Start Training

In [ ]:
train(
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    model=model,
    config=CONFIG,
    device=device,
    resume_from=resume_from,
)

## 9. Plot Training Loss

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = f"{CONFIG['metrics_dir']}/v2_training_history.json"
with open(history_path) as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]
losses = [h["loss"] for h in history]

plt.figure(figsize=(10, 5))
plt.plot(epochs, losses, "b-", linewidth=2, label="Train")
if "val_loss" in history[0]:
    val_losses = [h["val_loss"] for h in history]
    plt.plot(epochs, val_losses, "r-", linewidth=2, label="Val")
    plt.legend()
plt.xlabel("Epoch")
plt.ylabel("L1 Loss")
plt.title("DDPM v2 Training Loss")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Generate Synthetic MRIs

In [ ]:
from src.med_ddpm_v2.sample import generate_synthetic

# Find latest checkpoint
checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint_v2_epoch_*.pt")
best = max(checkpoints, key=lambda p: int(Path(p).stem.split('_')[-1]))
print(f"Generating from: {best}")

generate_synthetic(
    checkpoint_path=best,
    output_dir=CONFIG["synthetic_dir"],
    config=CONFIG,
)